# Experiment — can Bayesian tuning beat my baseline?

I have a baseline XGBoost model. **Hypothesis:** a Bayesian (Optuna TPE) search
over the hyperparameters will lower the RMSE.

This notebook runs that search. **The search in section 2 is the long-running
job** — each trial cross-validates a fresh set of hyperparameters. Raise
`N_TRIALS` to make it take longer (that's exactly what you want for the demo:
something that keeps running while you go and do other work).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import plotly.express as px
import xgboost as xgb
from sklearn.metrics import mean_squared_error

from backend.data import load_dataset, RANDOM_STATE
from backend.train import BASELINE_PARAMS

N_TRIALS = 100   # <-- raise this to make the search run longer

X_train, X_test, y_train, y_test = load_dataset()

def rmse(model):
    return float(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))

## 1. Baseline

Plain XGBoost with sensible default hyperparameters — the number to beat.

In [ ]:
baseline = xgb.XGBRegressor(**BASELINE_PARAMS, random_state=RANDOM_STATE, tree_method='hist')
baseline.fit(X_train, y_train)
baseline_rmse = rmse(baseline)
print(f'baseline RMSE: {baseline_rmse:.4f}')

## 2. Bayesian search  ⏳ (the long job)

This is the cell that takes a while. Watch `best` tick downwards as the TPE
sampler learns which regions of the hyperparameter space are promising.

*While this runs is exactly when the colourblind bug report lands — see the
runsheet.*

In [ ]:
from backend.tuning import bayesian_search

best_params = bayesian_search(X_train, y_train, n_trials=N_TRIALS)
best_params

## 3. Did it help?

In [ ]:
tuned = xgb.XGBRegressor(**best_params, random_state=RANDOM_STATE, tree_method='hist')
tuned.fit(X_train, y_train)
tuned_rmse = rmse(tuned)

print(f'baseline RMSE: {baseline_rmse:.4f}')
print(f'tuned    RMSE: {tuned_rmse:.4f}')
print(f'improvement:   {(baseline_rmse - tuned_rmse) / baseline_rmse * 100:.1f}%')

In [ ]:
px.bar(x=['baseline', 'tuned'], y=[baseline_rmse, tuned_rmse],
       title='RMSE: baseline vs Bayesian-tuned (lower is better)',
       labels={'x': 'model', 'y': 'RMSE'})

The tuned model wins. To ship it for the dashboard, `python -m backend.train
--mode tuned` runs the same search and saves the model atomically to `models/`.